# 02 — Classical ML Pipeline

Train TF-IDF-based pipelines: Naive Bayes, Logistic Regression, Linear SVM, XGBoost. Evaluate and compare using F1, accuracy, AUC-ROC, and confusion matrices.

In [ ]:
import sys
sys.path.insert(0, "..")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import load_imdb_data
from src.models import MODELS
from src.evaluate import compute_metrics, print_report, confusion_matrix_df, results_table

sns.set_theme(style="whitegrid")
%matplotlib inline


## Load Data

In [ ]:
train_df, test_df = load_imdb_data()

X_train, y_train = train_df["clean_text"], train_df["label"]
X_test,  y_test  = test_df["clean_text"],  test_df["label"]


## Train All Models

In [ ]:
all_metrics    = {}
trained_models = {}

for name, builder in MODELS.items():
    print(f"\nTraining: {name}")
    model = builder()

    t0 = time.time()
    model.fit(X_train, y_train)
    print(f"  Time: {time.time()-t0:.1f}s")

    y_pred  = model.predict(X_test)
    y_prob  = model.predict_proba(X_test)[:,1] if hasattr(model.named_steps["clf"], "predict_proba") else None

    print_report(name, y_test, y_pred)
    all_metrics[name]    = compute_metrics(y_test, y_pred, y_prob)
    trained_models[name] = (model, y_pred, y_prob)


## Results Table

In [ ]:
results_table(all_metrics)


## Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, (name, (model, y_pred, _)) in zip(axes.flatten(), trained_models.items()):
    cm = confusion_matrix_df(y_test, y_pred).values
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Pred Neg", "Pred Pos"],
                yticklabels=["True Neg", "True Pos"])
    ax.set_title(name, fontweight="bold")

plt.tight_layout()
plt.savefig("../results/confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.show()


## Metric Comparison Bar Chart

In [ ]:
df_results = results_table(all_metrics).reset_index()

metrics_to_plot = ["accuracy", "f1_weighted", "precision", "recall"]
x = np.arange(len(df_results))
width = 0.2

fig, ax = plt.subplots(figsize=(13, 5))
colors = ["#4c72b0", "#dd8452", "#55a868", "#c44e52"]

for i, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    ax.bar(x + i * width, df_results[metric], width, label=metric.replace("_", " ").title(), color=color)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(df_results["Model"], rotation=15, ha="right")
ax.set_ylim(0.80, 0.97)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — Classical ML")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("../results/model_comparison_bar.png", dpi=120, bbox_inches="tight")
plt.show()


## Top TF-IDF Features — Logistic Regression

In [ ]:
lr_model  = trained_models["Logistic Regression"][0]
tfidf     = lr_model.named_steps["tfidf"]
clf       = lr_model.named_steps["clf"]
feature_names = tfidf.get_feature_names_out()

top_n   = 20
coefs   = clf.coef_[0]
top_pos = np.argsort(coefs)[-top_n:][::-1]
top_neg = np.argsort(coefs)[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, indices, color, label in zip(
    axes, [top_pos, top_neg], ["#7bc4e0", "#e07b7b"], ["Most Positive", "Most Negative"]
):
    ax.barh(feature_names[indices][::-1], coefs[indices][::-1], color=color, edgecolor="white")
    ax.set_title(f"Top {top_n} {label} Features")
    ax.set_xlabel("Coefficient")

plt.tight_layout()
plt.savefig("../results/top_features_lr.png", dpi=120, bbox_inches="tight")
plt.show()
